# Gold Layer - Star Schema

Builds the gold-layer star schema for the Power BI dashboard:
- Conformed dimensions: dim_patient, dim_provider, dim_organization, dim_payer, dim_state, dim_date
- Fact tables: fact_encounters, fact_claims, fact_patient_utilization, fact_clinical_events
- Bridge: bridge_claim_diagnosis

Each dimension carries an integer surrogate key (`*_sk`) used by the fact tables
for joins. The original business keys are preserved on dimensions for
traceability back to silver but dropped from fact tables to keep them compact.

## 1. Imports and validation run setup

In [0]:
from pyspark.sql.functions import (
    array, struct, explode_outer, col, count, countDistinct, sum, avg, max, min,
    when, datediff, year, month, quarter, dayofmonth,
    dayofweek, date_format, explode, sequence, to_date, lit,
    row_number
)
from pyspark.sql.window import Window

import sys
sys.path.append('/Workspace/Shared/healthcare_demo')
from validation_helper import ValidationRun


In [0]:
run = ValidationRun(
    spark=spark,
    catalog="healthcare_demo",
    notebook_name="04_gold_star_schema",
    layer="gold"
)

print(f"Validation run started: {run.run_id}")

Validation run started: 12a49983-52d1-4c45-bba0-4935c5cf0eb9


## 2. Catalog and schemas

In [0]:
catalog_name = "healthcare_demo"
silver_schema = "healthcare_silver"
gold_schema = "healthcare_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")


DataFrame[]

## 3. Load silver tables

In [0]:
patients = spark.table(f"{catalog_name}.{silver_schema}.patients")
encounters = spark.table(f"{catalog_name}.{silver_schema}.encounters")
claims = spark.table(f"{catalog_name}.{silver_schema}.claims")
conditions = spark.table(f"{catalog_name}.{silver_schema}.conditions")
procedures = spark.table(f"{catalog_name}.{silver_schema}.procedures")
medications = spark.table(f"{catalog_name}.{silver_schema}.medications")
observations = spark.table(f"{catalog_name}.{silver_schema}.observations")
providers = spark.table(f"{catalog_name}.{silver_schema}.providers")
organizations = spark.table(f"{catalog_name}.{silver_schema}.organizations")
payers = spark.table(f"{catalog_name}.{silver_schema}.payers")


## 4. dim_patient

Builds dim_patient from silver patients. Adds `patient_sk` as the integer
surrogate key. `patient_id` is kept as the business key for traceability.

In [0]:
dim_patient = (
    patients
    .select(
        col("id").alias("patient_id"),
        col("birthdate"),
        col("deathdate"),
        col("gender"),
        col("race"),
        col("ethnicity"),
        col("city"),
        col("state"),
        col("county"),
        col("source_state")
    )
    .withColumn("birth_year", year(col("birthdate")))
    .withColumn(
        "patient_status",
        when(col("deathdate").isNotNull(), "Deceased").otherwise("Alive")
    )
    .dropDuplicates(["patient_id"])
)

# Surrogate key: deterministic ordering on patient_id so SK is stable across reruns
dim_patient = dim_patient.withColumn(
    "patient_sk",
    row_number().over(Window.orderBy("patient_id")).cast("int")
)

# Reorder columns: surrogate key first, then business key, then attributes
dim_patient = dim_patient.select(
    "patient_sk",
    "patient_id",
    "birthdate",
    "deathdate",
    "birth_year",
    "patient_status",
    "gender",
    "race",
    "ethnicity",
    "city",
    "county",
    "state",
    "source_state"
)

display(dim_patient.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


patient_sk,patient_id,birthdate,deathdate,birth_year,patient_status,gender,race,ethnicity,city,county,state,source_state
1,00035b9c-abd8-fad7-0044-4926c17048e3,1993-03-03,null,1993,Alive,F,White,Nonhispanic,New York,New York County,New York,New York
2,000494b1-6dac-9348-6489-898b6c4ff07f,1955-10-26,null,1955,Alive,F,White,Hispanic,Venice Gardens,Sarasota County,Florida,Florida
3,0008a7a6-506c-e7ad-dbcc-68cccc4833cf,1972-05-29,null,1972,Alive,M,Black,Hispanic,Austin,Travis County,Texas,Texas
4,0008caef-03a4-e8ae-6423-5a71b4605354,1975-11-10,null,1975,Alive,F,White,Nonhispanic,Warrington,Escambia County,Florida,Florida
5,000b7a68-1631-4432-3315-1f661beb5d04,1946-07-05,2022-12-26,1946,Deceased,M,White,Nonhispanic,Kingston,Ulster County,New York,New York
6,000cb105-0f6a-e089-2701-7fa5fdf7ec03,1954-04-06,null,1954,Alive,F,Asian,Nonhispanic,Elk Grove,Sacramento County,California,California
7,00162663-d9d1-4955-97bd-2a32662b9f4e,2002-06-12,null,2002,Alive,M,White,Nonhispanic,Palmdale,Los Angeles County,California,California
8,001727b6-dbde-2493-f71e-ea2cc0d6fe98,2025-08-22,null,2025,Alive,F,Asian,Hispanic,Turlock,Stanislaus County,California,California
9,0018dc8b-810e-f934-6a4c-263cf585f11b,1990-12-05,null,1990,Alive,M,Asian,Nonhispanic,Miami,Miami-Dade County,Florida,Florida
10,001b200b-2435-a52f-98ff-ce89f7f33a8e,1993-07-19,null,1993,Alive,M,Black,Nonhispanic,Perth,Fulton County,New York,New York


In [0]:
dim_patient.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_patient")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 5. dim_provider

In [0]:
dim_provider = (
    providers
    .select(
        col("id").alias("provider_id"),
        col("organization").alias("organization_id"),
        col("name").alias("provider_name"),
        col("gender").alias("provider_gender"),
        col("speciality").alias("provider_speciality"),
        col("city"),
        col("state"),
        col("source_state")
    )
    .dropDuplicates(["provider_id"])
)

dim_provider = dim_provider.withColumn(
    "provider_sk",
    row_number().over(Window.orderBy("provider_id")).cast("int")
)

# Reorder columns: surrogate key first, then business key, then attributes
dim_provider = dim_provider.select(
    "provider_sk",
    "provider_id",
    "organization_id",
    "provider_name",
    "provider_gender",
    "provider_speciality",
    "city",
    "state",
    "source_state"
)

display(dim_provider.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


provider_sk,provider_id,organization_id,provider_name,provider_gender,provider_speciality,city,state,source_state
1,0003c52f-4ddd-3f55-9f2f-8ac8f952a2ac,a5b047dc-3267-327a-8238-5b4899a7ddb6,Leticia253 Caraballo427,F,GENERAL PRACTICE,SANTA CLARA,CA,California
2,000467e5-a2df-39e9-b845-52621f783363,f9ad63a9-cf93-39ce-ac6d-35555a8e2824,Romeo514 Dach178,M,GENERAL PRACTICE,PASADENA,CA,California
3,0005f749-d8c1-3cf0-8380-b5034b6628e4,0de16e8e-88a9-3600-ad17-f3dafda8e867,Leopoldo762 White193,M,GENERAL PRACTICE,RANCHO CORDOVA,CA,California
4,000bb651-4f31-3fc4-846d-1fef3a5a867d,b7fdb5dd-0a05-3d40-8640-7e6c9657730a,Twyla619 McKenzie376,F,GENERAL PRACTICE,FT WORTH,TX,Texas
5,000c09cd-53a2-3267-9b0e-22d13122f8c8,13c5ac3e-aed5-3a59-96df-122927f03395,Christopher407 Hammes673,F,GENERAL PRACTICE,CARROLLTON,TX,Texas
6,0013b637-6709-3fda-8dd7-202c21d9c6fd,a1cd2a93-5078-380a-a98c-9bfa4c9411f9,Kizzie166 Flatley871,F,GENERAL PRACTICE,JAMISON,PA,Pennsylvania
7,0014347f-2592-3797-8988-fba36aab804f,03027b60-e8dd-351e-99a5-1d3e443b2417,Clifford177 Bahringer146,M,GENERAL PRACTICE,SAN DIEGO,CA,California
8,0017402c-ce9e-3239-ae3b-cc1d42e8ef21,c17fe56a-5b9a-3364-9737-0ecbc43bb647,Adelia946 Schaefer657,F,GENERAL PRACTICE,MIAMI LAKES,FL,Florida
9,0018ac7a-28b2-32fa-800a-064f7386625e,f35d463d-303e-362b-8722-96bf221545c8,Warren653 O'Keefe54,M,GENERAL PRACTICE,ABINGTON,PA,Pennsylvania
10,001a0024-5983-3941-877e-087e113a9466,0edd40f8-227e-344c-97cf-bbe3a6038c46,Shakita329 Moen819,F,GENERAL PRACTICE,OGDENSBURG,NY,New York


In [0]:
dim_provider.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_provider")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 6. dim_organization

In [0]:
dim_organization = (
    organizations
    .select(
        col("id").alias("organization_id"),
        col("name").alias("organization_name"),
        col("address"),
        col("city"),
        col("state"),
        col("zip"),
        col("phone"),
        col("source_state")
    )
    .dropDuplicates(["organization_id"])
)

dim_organization = dim_organization.withColumn(
    "organization_sk",
    row_number().over(Window.orderBy("organization_id")).cast("int")
)

# Reorder columns: surrogate key first, then business key, then attributes
dim_organization = dim_organization.select(
    "organization_sk",
    "organization_id",
    "organization_name",
    "address",
    "city",
    "state",
    "zip",
    "phone",
    "source_state"
)

display(dim_organization.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


organization_sk,organization_id,organization_name,address,city,state,zip,phone,source_state
1,00050420-af57-353e-82e8-72e465a20f1c,LOMPOC VALLEY MEDICAL CTR COMP CARE CTR D/P SNF,216 N THIRD STREET,LOMPOC,CA,934366104,8057363466,California
2,000eea38-0ce2-3ca5-a657-5981a3563152,HAINES CITY REHAB LLC,409 S 10TH ST,HAINES CITY,FL,338445603,8634228656,Florida
3,000f52d2-1726-3ec5-92c1-e8768cc280f0,GLEN COVE CENTER FOR NURSING AND REHABILITATION,6 MEDICAL PLAZA,GLEN COVE,NY,115422108,5166568000,New York
4,0011bd31-fd2e-3855-9f9b-917928cfddcb,FAMILY AND CHILDREN'S ASSOCIATION,510 HEMPSTEAD TPKE,HICKSVILLE,NY,118014230,5164376050,New York
5,001f3f2b-eb00-3373-b2ad-07a098355a46,BAY CENTER REHABILITATION LLC,1336 SAINT ANDREWS BLVD,PANAMA CITY,FL,324052798,8635330578,Florida
6,002600ad-aea8-3500-89f1-6771662e24ac,CALZADA PRIMARY CARE PA,2261 N UNIVERSITY DR,PEMBROKE PINES,FL,330243617,9549874900,Florida
7,002ae05b-d618-3385-8f68-dcfcb3a32f3b,GREEN HOME,37 CENTRAL AVE,WELLSBORO,PA,169011857,5707243131,Pennsylvania
8,002d3024-cf6a-354d-ad6c-bb1b5bdab988,KENNEDY POST ACUTE CARE CENTER,619 N. FAIRFAX AVE,LOS ANGELES,CA,900361714,3236510043,California
9,002ff8b0-2279-37d7-90b1-6d29b7ab0377,FOSSIL CREEK FAMILY MEDICAL CENTER,7510 N BEACH ST,FT WORTH,TX,761371505,8174981818,Texas
10,00315643-4515-31ae-a064-76977eb856f6,MORENO VALLEY SNF LLC,26940 E HOSPITAL ROAD,MORENO VALLEY,CA,925553923,8014479829,California


In [0]:
dim_organization.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_organization")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 7. dim_payer

In [0]:
dim_payer = (
    payers
    .select(
        col("id").alias("payer_id"),
        col("name").alias("payer_name"),
        col("ownership"),
        col("address"),
        col("city"),
        col("state_headquartered"),
        col("zip"),
        col("phone"),
        col("source_state")
    )
    .dropDuplicates(["payer_id"])
)

dim_payer = dim_payer.withColumn(
    "payer_sk",
    row_number().over(Window.orderBy("payer_id")).cast("int")
)

# Reorder columns: surrogate key first, then business key, then attributes
dim_payer = dim_payer.select(
    "payer_sk",
    "payer_id",
    "payer_name",
    "ownership",
    "address",
    "city",
    "state_headquartered",
    "zip",
    "phone",
    "source_state"
)

display(dim_payer)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


payer_sk,payer_id,payer_name,ownership,address,city,state_headquartered,zip,phone,source_state
1,0133f751-9229-3cfd-815f-b6d4979bdd6a,Aetna,PRIVATE,null,null,null,null,null,California
2,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,null,null,null,null,null,California
3,734afbd6-4794-363b-9bc0-6a3981533ed5,Anthem,PRIVATE,null,null,null,null,null,California
4,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,Cigna Health,PRIVATE,null,null,null,null,null,California
5,a735bf55-83e9-331a-899d-a82a60b9f60c,Medicare,GOVERNMENT,null,null,null,null,null,California
6,b046940f-1664-3047-bca7-dfa76be352a4,Blue Cross Blue Shield,PRIVATE,null,null,null,null,null,California
7,d18ef2e6-ef40-324c-be54-34a5ee865625,Dual Eligible,GOVERNMENT,null,null,null,null,null,California
8,d31fccc3-1767-390d-966a-22a5156f4219,UnitedHealthcare,PRIVATE,null,null,null,null,null,California
9,df166300-5a78-3502-a46a-832842197811,Medicaid,GOVERNMENT,null,null,null,null,null,California
10,e03e23c9-4df1-3eb6-a62d-f70f02301496,NO_INSURANCE,NO_INSURANCE,null,null,null,null,null,California


In [0]:
dim_payer.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_payer")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 8. dim_date

`date_key` (yyyyMMdd as int) is already a compact integer key, so no separate
surrogate is needed here.

In [0]:
date_bounds = encounters.select(
    min(to_date(col("start"))).alias("min_date"),
    max(to_date(col("start"))).alias("max_date")
).collect()[0]

min_date_value = date_bounds["min_date"]
max_date_value = date_bounds["max_date"]

dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{min_date_value}'),
            to_date('{max_date_value}'),
            interval 1 day
        )) AS date
    """)
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
)

display(dim_date.limit(20))

date,date_key,year,quarter,month,month_name,day,day_of_week,day_name
2021-01-01,20210101,2021,1,1,January,1,6,Friday
2021-01-02,20210102,2021,1,1,January,2,7,Saturday
2021-01-03,20210103,2021,1,1,January,3,1,Sunday
2021-01-04,20210104,2021,1,1,January,4,2,Monday
2021-01-05,20210105,2021,1,1,January,5,3,Tuesday
2021-01-06,20210106,2021,1,1,January,6,4,Wednesday
2021-01-07,20210107,2021,1,1,January,7,5,Thursday
2021-01-08,20210108,2021,1,1,January,8,6,Friday
2021-01-09,20210109,2021,1,1,January,9,7,Saturday
2021-01-10,20210110,2021,1,1,January,10,1,Sunday


In [0]:
dim_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_date")

## 9. dim_state

Five states only, no surrogate key needed — `state_key` (the state name) is
already a compact, stable key.

In [0]:
dim_state = spark.createDataFrame(
    [
        ("California",   "CA", "West",      "Pacific"),
        ("Florida",      "FL", "South",     "Eastern"),
        ("New York",     "NY", "Northeast", "Eastern"),
        ("Pennsylvania", "PA", "Northeast", "Eastern"),
        ("Texas",        "TX", "South",     "Central"),
    ],
    ["state_key", "state_abbreviation", "region", "time_zone"]
)

display(dim_state)

state_key,state_abbreviation,region,time_zone
California,CA,West,Pacific
Florida,FL,South,Eastern
New York,NY,Northeast,Eastern
Pennsylvania,PA,Northeast,Eastern
Texas,TX,South,Central


In [0]:
dim_state.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_state")

## 10. fact_encounters

Joins each dim to swap string business-key FKs (patient_id, provider_id, ...)
for compact integer SKs. The original string FKs are dropped after the join
to keep the fact table small.

In [0]:
# Lookup tables: business key -> surrogate key
patient_lookup = dim_patient.select("patient_id", "patient_sk")
provider_lookup = dim_provider.select("provider_id", "provider_sk")
organization_lookup = dim_organization.select("organization_id", "organization_sk")
payer_lookup = dim_payer.select("payer_id", "payer_sk")

fact_encounters = (
    encounters
    .select(
        col("id").alias("encounter_id"),
        col("patient").alias("patient_id"),
        col("provider").alias("provider_id"),
        col("organization").alias("organization_id"),
        col("payer").alias("payer_id"),
        to_date(col("start")).alias("encounter_start_date"),
        to_date(col("stop")).alias("encounter_stop_date"),
        col("encounterclass"),
        col("code").alias("encounter_code"),
        col("description").alias("encounter_description"),
        col("base_encounter_cost"),
        col("total_claim_cost"),
        col("payer_coverage"),
        col("reasoncode"),
        col("reasondescription"),
        col("source_state").alias("state_key")
    )
    .withColumn("encounter_date_key", date_format(col("encounter_start_date"), "yyyyMMdd").cast("int"))
    .withColumn("encounter_length_days", datediff(col("encounter_stop_date"), col("encounter_start_date")))
    # Swap string FKs for surrogate keys
    .join(patient_lookup, "patient_id", "left")
    .join(provider_lookup, "provider_id", "left")
    .join(organization_lookup, "organization_id", "left")
    .join(payer_lookup, "payer_id", "left")
)

# Generate encounter_sk for use by fact_claims and fact_clinical_events
fact_encounters = fact_encounters.withColumn(
    "encounter_sk",
    row_number().over(Window.orderBy("encounter_id")).cast("int")
)

# Drop the heavy string FK columns; keep encounter_id as the business key
fact_encounters = fact_encounters.drop(
    "patient_id", "provider_id", "organization_id", "payer_id"
)

# Reorder columns: own SK first, then dim FKs grouped, then degenerate dims,
# then business key, then dates, then measures, then derived
fact_encounters = fact_encounters.select(
    "encounter_sk",
    "patient_sk",
    "provider_sk",
    "organization_sk",
    "payer_sk",
    "state_key",
    "encounter_date_key",
    "encounter_id",
    "encounter_start_date",
    "encounter_stop_date",
    "encounter_length_days",
    "encounterclass",
    "encounter_code",
    "encounter_description",
    "reasoncode",
    "reasondescription",
    "base_encounter_cost",
    "total_claim_cost",
    "payer_coverage"
)

display(fact_encounters.limit(2))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


encounter_sk,patient_sk,provider_sk,organization_sk,payer_sk,state_key,encounter_date_key,encounter_id,encounter_start_date,encounter_stop_date,encounter_length_days,encounterclass,encounter_code,encounter_description,reasoncode,reasondescription,base_encounter_cost,total_claim_cost,payer_coverage
1,1,10868,10128,10,New York,20210824,00035b9c-abd8-fad7-00f7-e2ae134b36b6,2021-08-24,2021-08-24,0,Outpatient,308335008,Patient encounter procedure,389095005,Contraception care,138.42,22634.21,0.0
2,1,10910,14956,10,New York,20210512,00035b9c-abd8-fad7-3026-bc1fd95aa11a,2021-05-12,2021-05-12,0,Wellness,162673000,General examination of patient,null,null,171.78,1071.5,0.0


In [0]:
fact_encounters.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_encounters")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 11. fact_claims

Same pattern as fact_encounters: join dims to swap string FKs for SKs, then
drop the strings. Also generates `claim_sk` so bridge_claim_diagnosis can
join on the integer key instead of the 64-char hash.

In [0]:
# Reload fact_encounters from disk to pick up encounter_sk
fact_encounters_for_join = spark.table(f"{catalog_name}.{gold_schema}.fact_encounters") \
    .select("encounter_id", "encounter_sk")

# Primary and secondary payer need separate lookups (same source dim, different alias)
primary_payer_lookup = dim_payer.select(
    col("payer_id").alias("primary_payer_id"),
    col("payer_sk").alias("primary_payer_sk")
)
secondary_payer_lookup = dim_payer.select(
    col("payer_id").alias("secondary_payer_id"),
    col("payer_sk").alias("secondary_payer_sk")
)

fact_claims = (
    claims
    .select(
        col("id").alias("claim_id"),
        col("patientid").alias("patient_id"),
        col("providerid").alias("provider_id"),
        col("primarypatientinsuranceid").alias("primary_payer_id"),
        col("secondarypatientinsuranceid").alias("secondary_payer_id"),
        col("appointmentid").alias("encounter_id"),
        col("currentillnessdate"),
        col("servicedate"),
        col("status1"),
        col("status2"),
        col("statusp"),
        col("outstanding1"),
        col("outstanding2"),
        col("outstandingp"),
        col("lastbilleddate1"),
        col("lastbilleddate2"),
        col("lastbilleddatep"),
        col("source_state").alias("state_key")
    )
    .withColumn("service_date_key", date_format(to_date(col("servicedate")), "yyyyMMdd").cast("int"))
    # Swap string FKs for surrogate keys
    .join(patient_lookup, "patient_id", "left")
    .join(provider_lookup, "provider_id", "left")
    .join(primary_payer_lookup, "primary_payer_id", "left")
    .join(secondary_payer_lookup, "secondary_payer_id", "left")
    .join(fact_encounters_for_join, "encounter_id", "left")
)

# Generate claim_sk - bridge_claim_diagnosis will join on this
fact_claims = fact_claims.withColumn(
    "claim_sk",
    row_number().over(Window.orderBy("claim_id")).cast("int")
)

# Drop the heavy string FK columns; keep claim_id and encounter_id as business keys
fact_claims = fact_claims.drop(
    "patient_id", "provider_id", "primary_payer_id", "secondary_payer_id"
)

# Reorder columns: own SK first, then dim FKs grouped, then cross-fact FK,
# then degenerate dims, then business keys, then dates, then status, then measures
fact_claims = fact_claims.select(
    "claim_sk",
    "patient_sk",
    "provider_sk",
    "primary_payer_sk",
    "secondary_payer_sk",
    "encounter_sk",
    "state_key",
    "service_date_key",
    "claim_id",
    "encounter_id",
    "currentillnessdate",
    "servicedate",
    "lastbilleddate1",
    "lastbilleddate2",
    "lastbilleddatep",
    "status1",
    "status2",
    "statusp",
    "outstanding1",
    "outstanding2",
    "outstandingp"
)

display(fact_claims.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


claim_sk,patient_sk,provider_sk,primary_payer_sk,secondary_payer_sk,encounter_sk,state_key,service_date_key,claim_id,encounter_id,currentillnessdate,servicedate,lastbilleddate1,lastbilleddate2,lastbilleddatep,status1,status2,statusp,outstanding1,outstanding2,outstandingp
1,1,10868,null,null,11,New York,20251217,00035b9c-abd8-fad7-0e7e-6251690ad175,00035b9c-abd8-fad7-6dcc-1cb68e3fb6b7,2025-12-17T17:17:12Z,2025-12-17T17:17:12Z,2025-12-17T17:32:12Z,2025-12-17T17:32:12Z,2025-12-17T17:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
2,1,10868,null,null,16,New York,20231122,00035b9c-abd8-fad7-0e88-bbbc60b669cd,00035b9c-abd8-fad7-9f1b-21a920a53819,2023-11-22T17:17:12Z,2023-11-22T17:17:12Z,2023-11-22T17:32:12Z,2023-11-22T17:32:12Z,2023-11-22T17:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
3,1,10868,null,null,6,New York,20210821,00035b9c-abd8-fad7-1696-4007bf2e5f8a,00035b9c-abd8-fad7-4c30-1b32607b61f1,2021-08-21T17:17:12Z,2021-08-21T17:17:12Z,2021-08-21T17:32:12Z,2021-08-21T17:32:12Z,2021-08-21T17:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
4,1,10868,null,null,5,New York,20251207,00035b9c-abd8-fad7-16df-27b8bf650608,00035b9c-abd8-fad7-4a0c-fc180ec5ae63,2025-12-07T14:17:12Z,2025-12-07T14:17:12Z,2025-12-07T14:32:12Z,2025-12-07T14:32:12Z,2025-12-07T14:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
5,1,10868,null,null,13,New York,20210703,00035b9c-abd8-fad7-41f1-e9b42474c8e2,00035b9c-abd8-fad7-779c-54fc1a46f3af,2021-07-03T17:17:12Z,2021-07-03T17:17:12Z,2021-07-03T17:32:12Z,2021-07-03T17:32:12Z,2021-07-03T17:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
6,1,10868,null,null,5,New York,20251207,00035b9c-abd8-fad7-4a7d-6fb8556231bc,00035b9c-abd8-fad7-4a0c-fc180ec5ae63,2025-12-07T14:17:12Z,2025-12-07T14:17:12Z,2025-12-07T14:32:12Z,2025-12-07T14:32:12Z,2025-12-07T14:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
7,1,10868,9,null,9,New York,20220819,00035b9c-abd8-fad7-5c03-60b009d28a09,00035b9c-abd8-fad7-6232-caf025e1a477,2022-08-19T08:20:25Z,2022-08-19T08:20:25Z,2022-08-19T08:35:25Z,2022-08-19T08:35:25Z,2022-08-19T08:35:25Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
8,1,10868,9,null,9,New York,20220819,00035b9c-abd8-fad7-5da7-7fc921aa83f7,00035b9c-abd8-fad7-6232-caf025e1a477,2022-08-19T08:20:25Z,2022-08-19T08:20:25Z,2022-08-19T08:35:25Z,2022-08-19T08:35:25Z,2022-08-19T08:35:25Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
9,1,10868,null,null,14,New York,20231220,00035b9c-abd8-fad7-7574-0cc8ab59b950,00035b9c-abd8-fad7-7cdb-a35b46410405,2023-12-20T17:17:12Z,2023-12-20T17:17:12Z,2023-12-20T17:32:12Z,2023-12-20T17:32:12Z,2023-12-20T17:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0
10,1,10868,null,null,6,New York,20210821,00035b9c-abd8-fad7-789b-e92f6038283b,00035b9c-abd8-fad7-4c30-1b32607b61f1,2021-08-21T17:17:12Z,2021-08-21T17:17:12Z,2021-08-21T17:32:12Z,2021-08-21T17:32:12Z,2021-08-21T17:32:12Z,CLOSED,CLOSED,CLOSED,0.0,0.0,0.0


In [0]:
fact_claims.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_claims")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 12. bridge_claim_diagnosis

The bridge previously stored a full 64-character claim_id hash on every row,
making it the second-largest table in the model. Now joins to fact_claims to
get `claim_sk` and drops the string `claim_id` entirely. This is the biggest
single size win in gold.

In [0]:
# Build an array of (position, code) pairs from the 8 diagnosis columns
diagnosis_columns = [
    ("diagnosis1", 1),
    ("diagnosis2", 2),
    ("diagnosis3", 3),
    ("diagnosis4", 4),
    ("diagnosis5", 5),
    ("diagnosis6", 6),
    ("diagnosis7", 7),
    ("diagnosis8", 8),
]

diagnosis_struct_array = array(*[
    struct(lit(position).alias("diagnosis_position"), col(name).alias("diagnosis_code"))
    for name, position in diagnosis_columns
])

# Reload fact_claims from disk to pick up claim_sk
fact_claims_for_join = spark.table(f"{catalog_name}.{gold_schema}.fact_claims") \
    .select("claim_id", "claim_sk")

bridge_claim_diagnosis = (
    claims
    .select(
        col("id").alias("claim_id"),
        col("source_state").alias("state_key"),
        diagnosis_struct_array.alias("diagnosis_array")
    )
    .withColumn("diagnosis", explode(col("diagnosis_array")))
    .select(
        col("claim_id"),
        col("state_key"),
        col("diagnosis.diagnosis_position"),
        col("diagnosis.diagnosis_code")
    )
    .filter(col("diagnosis_code").isNotNull())
    # Swap the heavy claim_id string for the compact claim_sk integer
    .join(fact_claims_for_join, "claim_id", "left")
    .drop("claim_id")
)

# Reorder columns: surrogate key first, then degenerate dim, then position, then code
bridge_claim_diagnosis = bridge_claim_diagnosis.select(
    "claim_sk",
    "state_key",
    "diagnosis_position",
    "diagnosis_code"
)

display(bridge_claim_diagnosis.limit(20))

claim_sk,state_key,diagnosis_position,diagnosis_code
10312,New York,2,18718003
43720,California,1,389095005
125874,New York,1,46177005
152793,New York,1,431857002
158721,Florida,2,73595000
219431,California,1,73595000
244971,Texas,1,609328004
341677,California,1,185347001
344808,California,1,103697008
362810,California,1,185347001


In [0]:
bridge_claim_diagnosis.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.bridge_claim_diagnosis")

## 13. fact_patient_utilization

In [0]:
encounter_counts = (
    encounters
    .groupBy(col("patient").alias("patient_id"))
    .agg(countDistinct("id").alias("total_encounters"))
)

condition_counts = (
    conditions
    .groupBy(col("patient").alias("patient_id"))
    .agg(count("*").alias("total_conditions"))
)

procedure_counts = (
    procedures
    .groupBy(col("patient").alias("patient_id"))
    .agg(count("*").alias("total_procedures"))
)

medication_counts = (
    medications
    .groupBy(col("patient").alias("patient_id"))
    .agg(count("*").alias("total_medications"))
)

observation_counts = (
    observations
    .groupBy(col("patient").alias("patient_id"))
    .agg(count("*").alias("total_observations"))
)

In [0]:
fact_patient_utilization = (
    dim_patient
    .select("patient_id", "patient_sk", col("source_state").alias("state_key"))
    .join(encounter_counts, "patient_id", "left")
    .join(condition_counts, "patient_id", "left")
    .join(procedure_counts, "patient_id", "left")
    .join(medication_counts, "patient_id", "left")
    .join(observation_counts, "patient_id", "left")
    .fillna(0, subset=[
        "total_encounters",
        "total_conditions",
        "total_procedures",
        "total_medications",
        "total_observations"
    ])
    # Drop the business key; patient_sk is the FK used by Power BI
    .drop("patient_id")
)

# Reorder columns: surrogate key first, then degenerate dim, then measures
fact_patient_utilization = fact_patient_utilization.select(
    "patient_sk",
    "state_key",
    "total_encounters",
    "total_conditions",
    "total_procedures",
    "total_medications",
    "total_observations"
)

display(fact_patient_utilization.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


patient_sk,state_key,total_encounters,total_conditions,total_procedures,total_medications,total_observations
1,New York,17,17,73,3,39
2,Florida,31,38,122,37,519
3,Texas,15,24,49,17,234
4,Florida,8,19,31,1,143
5,New York,15,21,95,20,129
6,California,22,22,65,0,233
7,California,7,16,46,7,174
8,California,3,3,2,0,44
9,Florida,27,29,63,10,260
10,New York,7,12,21,0,99


In [0]:
fact_patient_utilization.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_patient_utilization")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 14. fact_clinical_events

In [0]:
condition_events = (
    conditions
    .select(
        col("patient").alias("patient_id"),
        col("encounter").alias("encounter_id"),
        to_date(col("start")).alias("event_date"),
        col("code").alias("event_code"),
        col("description").alias("event_description"),
        col("source_state").alias("state_key")
    )
    .withColumn("event_type", lit("condition"))
)

procedure_events = (
    procedures
    .select(
        col("patient").alias("patient_id"),
        col("encounter").alias("encounter_id"),
        to_date(col("start")).alias("event_date"),
        col("code").alias("event_code"),
        col("description").alias("event_description"),
        col("source_state").alias("state_key")
    )
    .withColumn("event_type", lit("procedure"))
)

medication_events = (
    medications
    .select(
        col("patient").alias("patient_id"),
        col("encounter").alias("encounter_id"),
        to_date(col("start")).alias("event_date"),
        col("code").alias("event_code"),
        col("description").alias("event_description"),
        col("source_state").alias("state_key")
    )
    .withColumn("event_type", lit("medication"))
)

observation_events = (
    observations
    .select(
        col("patient").alias("patient_id"),
        col("encounter").alias("encounter_id"),
        to_date(col("date")).alias("event_date"),
        col("code").alias("event_code"),
        col("description").alias("event_description"),
        col("source_state").alias("state_key")
    )
    .withColumn("event_type", lit("observation"))
)

fact_clinical_events = (
    condition_events
    .unionByName(procedure_events)
    .unionByName(medication_events)
    .unionByName(observation_events)
    .withColumn("event_date_key", date_format(col("event_date"), "yyyyMMdd").cast("int"))
    # Swap string FKs for surrogate keys
    .join(patient_lookup, "patient_id", "left")
    .join(fact_encounters_for_join, "encounter_id", "left")
    # Drop the heavy string FKs
    .drop("patient_id", "encounter_id")
)

# Reorder columns: dim SK FKs first, then degenerate dims, then event details
fact_clinical_events = fact_clinical_events.select(
    "patient_sk",
    "encounter_sk",
    "state_key",
    "event_date_key",
    "event_type",
    "event_date",
    "event_code",
    "event_description"
)

display(fact_clinical_events.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


patient_sk,encounter_sk,state_key,event_date_key,event_type,event_date,event_code,event_description
6,null,California,20170621,condition,2017-06-21,239873007,Osteoarthritis of knee
6,88,California,20240412,condition,2024-04-12,68496003,Polyp of colon
6,null,California,19910702,condition,1991-07-02,162864005,Body mass index 30+ - obesity
6,94,California,20230725,condition,2023-07-25,66383009,Gingivitis
6,94,California,20230725,condition,2023-07-25,73595000,Stress
6,94,California,20230725,condition,2023-07-25,741062008,Not in labor force
6,94,California,20230725,condition,2023-07-25,80583007,Severe anxiety (panic)
6,null,California,19990817,condition,1999-08-17,161744009,Past pregnancy history of miscarriage
6,96,California,20250805,condition,2025-08-05,160903007,Full-time employment
6,96,California,20250805,condition,2025-08-05,314529007,Medication review due


In [0]:
fact_clinical_events.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_clinical_events")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 15. Validation

Row counts, PK uniqueness, FK integrity. Updated to validate the integer
surrogate-key FKs that fact tables now use.

In [0]:
star_schema_tables = [
    "dim_patient",
    "dim_provider",
    "dim_organization",
    "dim_payer",
    "dim_state",
    "dim_date",
    "fact_encounters",
    "fact_claims",
    "bridge_claim_diagnosis",
    "fact_patient_utilization",
    "fact_clinical_events"
]

validation_rows = []

for table in star_schema_tables:
    df = spark.table(f"{catalog_name}.{gold_schema}.{table}")
    validation_rows.append((table, df.count(), len(df.columns)))

validation_df = spark.createDataFrame(
    validation_rows,
    ["table_name", "row_count", "column_count"]
)

display(validation_df)

# Record non-empty checks for every gold table
for row in validation_df.collect():
    table_name = row["table_name"]
    row_count = row["row_count"]
    is_passing = (row_count > 0)

    run.add_result(
        check_id=f"gold.{table_name}.non_empty",
        check_name=f"Gold table {table_name} must be non-empty",
        check_category="row_count",
        target_table=table_name,
        severity="error",
        expected=">0",
        actual=str(row_count),
        status="passed" if is_passing else "failed",
        message="" if is_passing else "Gold table is empty — gold pipeline broken"
    )

print(f"Recorded {validation_df.count()} non-empty assertions.")

table_name,row_count,column_count
dim_patient,22817,13
dim_provider,15449,9
dim_organization,15430,9
dim_payer,10,10
dim_state,5,4
dim_date,1826,9
fact_encounters,459785,19
fact_claims,850177,21
bridge_claim_diagnosis,1240662,4
fact_patient_utilization,22817,7


Recorded 11 non-empty assertions.


In [0]:
# Helper for FK orphan counting in gold
def gold_fk_orphan_count(child_df, fk_col, parent_df, pk_col):
    """Count rows in child_df where fk_col has no matching pk_col in parent_df. Nulls excluded."""
    return (
        child_df
        .filter(col(fk_col).isNotNull())
        .join(
            parent_df.select(col(pk_col).alias("_parent_pk")).dropDuplicates(),
            col(fk_col) == col("_parent_pk"),
            "left_anti"
        )
        .count()
    )

In [0]:
# Every dim's primary key must be unique - check both business key and surrogate key
dim_pks = {
    "dim_patient":      ("patient_id",      "patient_sk"),
    "dim_provider":     ("provider_id",     "provider_sk"),
    "dim_organization": ("organization_id", "organization_sk"),
    "dim_payer":        ("payer_id",        "payer_sk"),
    "dim_state":        ("state_key",       None),
    "dim_date":         ("date_key",        None),
}

for dim_table, (business_key, surrogate_key) in dim_pks.items():
    df = spark.table(f"{catalog_name}.{gold_schema}.{dim_table}")

    # Business key check
    dup_count = (
        df.groupBy(business_key)
          .count()
          .filter(col("count") > 1)
          .count()
    )

    run.add_result(
        check_id=f"gold.{dim_table}.pk_uniqueness",
        check_name=f"PK uniqueness for {dim_table}.{business_key}",
        check_category="pk_uniqueness",
        target_table=dim_table,
        target_column=business_key,
        severity="error",
        expected="0",
        actual=str(dup_count),
        status="passed" if dup_count == 0 else "failed",
        message="" if dup_count == 0 else f"{dup_count} duplicate PKs in dimension"
    )

    # Surrogate key check (only for dims that have one)
    if surrogate_key is not None:
        sk_dup_count = (
            df.groupBy(surrogate_key)
              .count()
              .filter(col("count") > 1)
              .count()
        )

        run.add_result(
            check_id=f"gold.{dim_table}.sk_uniqueness",
            check_name=f"SK uniqueness for {dim_table}.{surrogate_key}",
            check_category="pk_uniqueness",
            target_table=dim_table,
            target_column=surrogate_key,
            severity="error",
            expected="0",
            actual=str(sk_dup_count),
            status="passed" if sk_dup_count == 0 else "failed",
            message="" if sk_dup_count == 0 else f"{sk_dup_count} duplicate SKs in dimension"
        )

print(f"Recorded {len(dim_pks)} PK uniqueness checks (plus surrogate-key checks where applicable).")

Recorded 6 PK uniqueness checks (plus surrogate-key checks where applicable).


In [0]:
# Bridge invariants
bridge_df = spark.table(f"{catalog_name}.{gold_schema}.bridge_claim_diagnosis")

# 1. diagnosis_position must be between 1 and 8
invalid_position_count = bridge_df.filter(
    (col("diagnosis_position") < 1) | (col("diagnosis_position") > 8)
).count()

run.add_result(
    check_id="gold.bridge_claim_diagnosis.position_range",
    check_name="diagnosis_position must be between 1 and 8",
    check_category="domain_check",
    target_table="bridge_claim_diagnosis",
    target_column="diagnosis_position",
    severity="error",
    expected="0",
    actual=str(invalid_position_count),
    status="passed" if invalid_position_count == 0 else "failed",
    message="" if invalid_position_count == 0 else f"{invalid_position_count} rows with out-of-range position"
)

# 2. diagnosis_code must not be null (bridge filters nulls at build, this is a re-check)
null_code_count = bridge_df.filter(col("diagnosis_code").isNull()).count()

run.add_result(
    check_id="gold.bridge_claim_diagnosis.code_not_null",
    check_name="diagnosis_code must not be null",
    check_category="null_check",
    target_table="bridge_claim_diagnosis",
    target_column="diagnosis_code",
    severity="error",
    expected="0",
    actual=str(null_code_count),
    status="passed" if null_code_count == 0 else "failed",
    message="" if null_code_count == 0 else f"{null_code_count} null diagnosis codes (bridge filter failed)"
)

print("Recorded 2 bridge invariant checks.")

Recorded 2 bridge invariant checks.


In [0]:
# Load gold tables once for the FK checks
g = lambda t: spark.table(f"{catalog_name}.{gold_schema}.{t}")

fact_encounters_df  = g("fact_encounters")
fact_claims_df      = g("fact_claims")
fact_util_df        = g("fact_patient_utilization")
fact_events_df      = g("fact_clinical_events")
bridge_df           = g("bridge_claim_diagnosis")
dim_patient_df      = g("dim_patient")
dim_provider_df     = g("dim_provider")
dim_org_df          = g("dim_organization")
dim_payer_df        = g("dim_payer")
dim_state_df        = g("dim_state")
dim_date_df         = g("dim_date")

# FK checks now validate the integer surrogate-key relationships
# Each entry: (fact_table_name, fact_df, fact_fk_col, dim_table_name, dim_df, dim_pk_col, severity, note)
fk_checks_to_run = [
    # fact_encounters - all FKs are now SKs except state_key and encounter_date_key
    ("fact_encounters", fact_encounters_df, "patient_sk",         "dim_patient",      dim_patient_df,  "patient_sk",      "error", ""),
    ("fact_encounters", fact_encounters_df, "provider_sk",        "dim_provider",     dim_provider_df, "provider_sk",     "error", ""),
    ("fact_encounters", fact_encounters_df, "organization_sk",    "dim_organization", dim_org_df,      "organization_sk", "error", ""),
    ("fact_encounters", fact_encounters_df, "payer_sk",           "dim_payer",        dim_payer_df,    "payer_sk",        "error", ""),
    ("fact_encounters", fact_encounters_df, "state_key",          "dim_state",        dim_state_df,    "state_key",       "error", ""),
    ("fact_encounters", fact_encounters_df, "encounter_date_key", "dim_date",         dim_date_df,     "date_key",        "error", ""),

    # fact_claims
    ("fact_claims", fact_claims_df, "patient_sk",         "dim_patient",     dim_patient_df,     "patient_sk",     "error", ""),
    ("fact_claims", fact_claims_df, "provider_sk",        "dim_provider",    dim_provider_df,    "provider_sk",    "error", ""),
    ("fact_claims", fact_claims_df, "primary_payer_sk",   "dim_payer",       dim_payer_df,       "payer_sk",       "error", ""),
    ("fact_claims", fact_claims_df, "secondary_payer_sk", "dim_payer",       dim_payer_df,       "payer_sk",       "error", "Nulls excluded (no secondary coverage)"),
    ("fact_claims", fact_claims_df, "state_key",          "dim_state",       dim_state_df,       "state_key",      "error", ""),
    ("fact_claims", fact_claims_df, "service_date_key",   "dim_date",        dim_date_df,        "date_key",       "error", ""),
    ("fact_claims", fact_claims_df, "encounter_sk",       "fact_encounters", fact_encounters_df, "encounter_sk",   "error", "Cross-fact FK"),

    # fact_patient_utilization
    ("fact_patient_utilization", fact_util_df, "patient_sk", "dim_patient", dim_patient_df, "patient_sk", "error", ""),
    ("fact_patient_utilization", fact_util_df, "state_key",  "dim_state",   dim_state_df,   "state_key",  "error", ""),

    # fact_clinical_events
    ("fact_clinical_events", fact_events_df, "patient_sk",     "dim_patient",     dim_patient_df,     "patient_sk",   "error", ""),
    ("fact_clinical_events", fact_events_df, "state_key",      "dim_state",       dim_state_df,       "state_key",    "error", ""),
    ("fact_clinical_events", fact_events_df, "event_date_key", "dim_date",        dim_date_df,        "date_key",     "info",  "Soft FK — state-table events may have dates outside the analytical window"),
    # Soft FK: conditions in fact_clinical_events may reference encounters outside the analysis window
    ("fact_clinical_events", fact_events_df, "encounter_sk",   "fact_encounters", fact_encounters_df, "encounter_sk", "info",  "Soft FK — state-table events may reference out-of-window encounters"),

    # bridge - now joins on claim_sk instead of claim_id
    ("bridge_claim_diagnosis", bridge_df, "claim_sk",  "fact_claims", fact_claims_df, "claim_sk",  "error", ""),
    ("bridge_claim_diagnosis", bridge_df, "state_key", "dim_state",   dim_state_df,   "state_key", "error", ""),
]

fk_check_results = []

for fact_table, fact_df, fact_fk, dim_table, dim_df, dim_pk, severity, note in fk_checks_to_run:
    orphan_count = gold_fk_orphan_count(fact_df, fact_fk, dim_df, dim_pk)

    if severity == "info":
        status = "passed"
        msg = note
    else:
        status = "passed" if orphan_count == 0 else "failed"
        msg = note if orphan_count == 0 else f"{orphan_count} orphan FK references"

    run.add_result(
        check_id=f"gold.{fact_table}.fk_{dim_table}.{fact_fk}",
        check_name=f"FK {fact_table}.{fact_fk} -> {dim_table}.{dim_pk}",
        check_category="fk_integrity",
        target_table=fact_table,
        target_column=fact_fk,
        severity=severity,
        expected="0" if severity == "error" else ">=0 (soft FK)",
        actual=str(orphan_count),
        status=status,
        message=msg
    )

    fk_check_results.append((fact_table, fact_fk, dim_table, dim_pk, orphan_count, severity))

fk_check_results_df = spark.createDataFrame(
    fk_check_results,
    ["fact_table", "fk_column", "dim_table", "dim_pk", "orphan_count", "severity"]
)
display(fk_check_results_df)
print(f"Recorded {len(fk_checks_to_run)} FK integrity checks.")


fact_table,fk_column,dim_table,dim_pk,orphan_count,severity
fact_encounters,patient_sk,dim_patient,patient_sk,0,error
fact_encounters,provider_sk,dim_provider,provider_sk,0,error
fact_encounters,organization_sk,dim_organization,organization_sk,0,error
fact_encounters,payer_sk,dim_payer,payer_sk,0,error
fact_encounters,state_key,dim_state,state_key,0,error
fact_encounters,encounter_date_key,dim_date,date_key,0,error
fact_claims,patient_sk,dim_patient,patient_sk,0,error
fact_claims,provider_sk,dim_provider,provider_sk,0,error
fact_claims,primary_payer_sk,dim_payer,payer_sk,0,error
fact_claims,secondary_payer_sk,dim_payer,payer_sk,0,error


Recorded 21 FK integrity checks.


## 16. Persist results and assert

In [0]:
summary = run.commit()

print(f"Run ID:         {summary['run_id']}")
print(f"Status:         {summary['status']}")
print(f"Total checks:   {summary['total']}")
print(f"  Passed:       {summary['passed']}")
print(f"  Warned:       {summary['warned']}")
print(f"  Failed:       {summary['failed']}")
print(f"Duration:       {summary['duration_seconds']:.2f}s")

Run ID:         12a49983-52d1-4c45-bba0-4935c5cf0eb9
Status:         passed
Total checks:   44
  Passed:       44
  Warned:       0
  Failed:       0
Duration:       115.48s


In [0]:
run.assert_no_failures()

## 17. Power BI relationships

In Power BI, build relationships using the **surrogate keys (`*_sk`)** rather
than the original business keys:

| From (fact) | To (dim) | Join columns |
|---|---|---|
| fact_encounters.patient_sk | dim_patient.patient_sk | one-to-many |
| fact_encounters.provider_sk | dim_provider.provider_sk | one-to-many |
| fact_encounters.organization_sk | dim_organization.organization_sk | one-to-many |
| fact_encounters.payer_sk | dim_payer.payer_sk | one-to-many |
| fact_encounters.state_key | dim_state.state_key | one-to-many |
| fact_encounters.encounter_date_key | dim_date.date_key | one-to-many |
| fact_claims.patient_sk | dim_patient.patient_sk | one-to-many |
| fact_claims.provider_sk | dim_provider.provider_sk | one-to-many |
| fact_claims.primary_payer_sk | dim_payer.payer_sk | one-to-many |
| fact_claims.secondary_payer_sk | dim_payer.payer_sk | inactive (alt path) |
| fact_claims.state_key | dim_state.state_key | one-to-many |
| fact_claims.service_date_key | dim_date.date_key | one-to-many |
| fact_claims.encounter_sk | fact_encounters.encounter_sk | one-to-many |
| bridge_claim_diagnosis.claim_sk | fact_claims.claim_sk | many-to-one |
| bridge_claim_diagnosis.state_key | dim_state.state_key | many-to-one |
| fact_patient_utilization.patient_sk | dim_patient.patient_sk | one-to-many |
| fact_patient_utilization.state_key | dim_state.state_key | many-to-one |
| fact_clinical_events.patient_sk | dim_patient.patient_sk | many-to-one |
| fact_clinical_events.encounter_sk | fact_encounters.encounter_sk | many-to-one |
| fact_clinical_events.state_key | dim_state.state_key | many-to-one |
| fact_clinical_events.event_date_key | dim_date.date_key | many-to-one |

Business keys (patient_id, claim_id, encounter_id, etc.) remain available on
the dimension/fact tables for display, traceability, or debugging. They are
not used for joins.

## 18. Completion summary

In [0]:
# Sanity check 1: row counts for fact_claims and bridge
fact_claims_count = spark.table(f"{catalog_name}.{gold_schema}.fact_claims").count()
bridge_count = spark.table(f"{catalog_name}.{gold_schema}.bridge_claim_diagnosis").count()

print(f"fact_claims row count:            {fact_claims_count:,}")
print(f"bridge_claim_diagnosis row count: {bridge_count:,}")
print(f"Average diagnoses per claim:      {bridge_count / fact_claims_count:.2f}")


fact_claims row count:            850,177
bridge_claim_diagnosis row count: 1,240,662
Average diagnoses per claim:      1.46
